# Глава 7. Оценка эффективности RAG‑систем

## 7.1. Метрики качества для RAG

Оценить качество RAG-системы сложнее, чем обычной модели. Нужно проверять не только итоговый ответ, но и поиск, генерацию, а также их совместную работу.

С развитием RAG появились специальные фреймворки для оценки — RAGAS, TruLens, LlamaIndex. Каждый из них делает упор на разные части системы.

Оценка строится на трёх измерениях:
1. Качество поиска (retrieval quality).
2. Качество генерации (generation quality).
3. Итоговая эффективность системы.

Для каждого измерения нужны свои метрики, адаптированные под архитектуру RAG.

### Метрики качества поиска

**Hit Rate** — доля запросов, где в топ‑K результатов есть хотя бы один релевантный документ.

**Mean Reciprocal Rank (MRR)** учитывает позицию первого релевантного документа. Чем он ниже, тем хуже оценка.

$\text{MRR} = \frac{1}{Q} \sum_{i=1}^{Q} \frac{1}{\text{rank}_i}$

где Q — количество запросов, ${\text{rank}_i}$ — позиция первого релевантного документа для запроса i.

**Normalized Discounted Cumulative Gain (NDCG)** оценивает качество ранжирования, учитывая и релевантность, и позицию.

$\text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$

где $\text{DCG@K} = \sum_{i=1}^{K} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)}$, ${rel}_i$ — релевантность документа на позиции i.

### Основные метрики фреймворка RAGAS

**Context Precision** — качество ранжирования релевантных фрагментов в извлечённом контексте. Языковая модель оценивает релевантность каждого фрагмента запросу.

$\text{Context Precision} = \frac{\sum_{k=1}^{K} \text{Precision@K} \times v_k}{\text{Количество релевантных элементов в топ‑K}}$, 
где $v_k$ — бинарный индикатор релевантности элемента на позиции $k$.


**Context Recall** — полнота извлечения релевантной информации. Сравнивает извлечённый контекст с эталонным ответом.

$\text{Context Recall} = \frac{\text{Количество предложений в базовой истине, поддерживаемых контекстом}}{\text{Общее количество предложений в базовой истине}}$

**Faithfulness (верность)** — фактическая точность ответа относительно предоставленного контекста. Предотвращает галлюцинации. Ответ разбивается на атомарные утверждения, каждое проверяется по контексту.

$\text{Faithfulness} = \frac{\text{Количество утверждений в ответе, поддерживаемых контекстом}}{\text{Общее количество утверждений в ответе}}$

**Answer Relevancy** — соответствие ответа исходному запросу. Из ответа генерируются псевдовопросы и вычисляется их семантическое сходство с оригинальным запросом.

$\text{Answer Relevancy} = \frac{1}{N} \sum_{i=1}^{N} \cos(\text{emb}(question), \text{emb}(generated\_question_i))$, где $N$ — количество сгенерированных псевдовопросов.

### Дополнительные метрики оценки

**Answer Semantic Similarity** — семантическая близость сгенерированного и эталонного ответов. Вычисляется через косинусное сходство эмбеддингов, полученных предобученной моделью.

**Answer Correctness** — взвешенная комбинация семантического и фактологического сходства.

$\text{Answer Correctness} = w_1 \times \text{Semantic Similarity} + w_2 \times \text{Factual Similarity}$

**Exact Match** — строгая бинарная метрика: 1 при полном совпадении ответа с эталоном, 0 в остальных случаях.

**BLEU, ROUGE, METEOR** — метрики, изначально созданные для машинного перевода и суммаризации. Адаптируются под генерацию текста в RAG-системах.

### Контекстуальные метрики

**Context Relevancy** — релевантность каждого фрагмента контекста исходному запросу. Большая языковая модель классифицирует предложения контекста.

**Context Entity Recall** — полнота извлечения именованных сущностей. Важна для фактографических запросов, где нужно найти конкретные объекты, персоны, места.

**Noise Robustness** — устойчивость к нерелевантному контексту. Измеряет падение качества при добавлении шума к релевантным документам.

### Метрики латентности и производительности

**End-to-End Latency** — общее время от запроса до получения ответа. Критически важно для интерактивных применений.

**Retrieval Latency** — время поиска документов. Помогает найти узкие места в поисковом компоненте.

**Generation Latency** — время генерации ответа моделью при фиксированном контексте.

**Throughput** — количество запросов в единицу времени при заданных ресурсах.

### Холистические метрики

**User Satisfaction** — итоговая метрика успеха. Измеряется через явную обратную связь или неявные сигналы: время взаимодействия, повторные запросы.

**Task Success Rate** — доля запросов, где система дала полезный и действенный ответ. Особенно важна для задачно-ориентированных систем.

**Hallucination Rate** — частота генерации фактически неверной информации. Критична, когда точность на первом месте.

**Citation Accuracy** — корректность атрибуции информации для систем со ссылками на источники. Доля утверждений, правильно связанных со своими источниками.

**Итог.** Оценка RAG-систем требует баланса метрик под конкретные требования. Одна метрика не даёт полной картины — только комбинация позволяет объективно оценить эффективность. Набор метрик должен соответствовать бизнес-целям и ожиданиям пользователей.

## 7.2. Методы тестирования и валидации

Тестировать RAG-системы нужно послойно — от отдельных частей до всей системы целиком. Это сложнее, чем в обычной разработке, потому что языковые модели работают недетерминированно (выдают разные результаты на один запрос).

Подход включает четыре уровня:
1. **Модульное тестирование** — проверка каждого компонента по отдельности.
2. **Интеграционное тестирование** — проверка связей между компонентами.
3. **Комплексное тестирование** — проверка пользовательских сценариев.
4. **Нагрузочное тестирование** — проверка производительности под нагрузкой.

Для каждого уровня нужны свои методы и инструменты.

### Модульное тестирование компонентов RAG

**Модульное тестирование** проверяет каждую часть RAG-системы отдельно:

- **Ретривер** — насколько хорошо ищет документы.  
  Тесты: точные совпадения, поиск по смыслу (синонимы), устойчивость к опечаткам, крайние случаи (пустой или слишком длинный запрос). Для каждого теста готовят эталонные результаты. Данные — синтетические и реальные.

- **Генератор** — качество ответа при зафиксированном контексте.  
  Проверяют: опору на контекст, связность текста, отсутствие выдумок (галлюцинаций), нужную длину ответа. Используют автоматические метрики и ручную оценку на важных примерах.

- **Эмбеддинг-модель** — стабильность векторных представлений.  
  Проверяют: одинаковость векторов для одного содержания, реакцию на мелкие правки текста, скорость векторизации, правильную размерность векторов. Обязательно тестируют на разных языках и в разных предметных областях.

### Интеграционное тестирование взаимодействий

**Интеграционное тестирование** — проверка связки «ретривер → генератор» и передачи данных между ними.

**Что проверяют:**

- **RAG-пайплайн** на запросах разной сложности:  
  простые (однозначный ответ), многошаговые (комплексный анализ), противоречивые или неполные данные (устойчивость).

- **API** — внешние интерфейсы:  
  форматы ввода/вывода, невалидные запросы, лимиты токенов, HTTP-коды, время ответа, обработка тайм-аутов.

- **Память и состояние** (для диалоговых систем):  
  удержание контекста, обновление предпочтений, управление сессиями, поведение при переполнении контекстного окна. Тестируется на основе состояний.

### Комплексное тестирование пользовательских сценариев

**Комплексное тестирование** проверяет систему на реальных сценариях, включая пользовательский опыт.

**Что входит:**

- **Сценарное тестирование** — создаются тестовые персоны:  
  новички (понятность и объяснения), эксперты (глубина и точность сложных ответов), продвинутые пользователи (расширенный функционал и скорость).

- **Тестирование на реальных данных** — используют близкие к жизни датасеты.  
  Приватность обеспечивают маскированием данных, синтетическими данными или тестовыми «песочницами».

- **Регрессионное тестирование** — набор обязательных тестов, проходимых после каждого обновления. Особый контроль — за изменениями моделей (могут неожиданно ухудшить качество).

### A/B‑тестирование и эксперименты

**A/B-тестирование RAG** учитывает, что случайность языковых моделей может скрыть реальные различия между вариантами.

**Как строится:**

- **Дизайн экспериментов** — сравнивают: модели ретриверов, стратегии ранжирования, варианты промптов, размеры контекстного окна. У каждого эксперимента — чёткая гипотеза и измеримый результат.

- **Статистическая значимость** — анализируют разброс ответов модели, различия между пользователями, временны́е эффекты. Для точности применяют стратифицированную выборку и блоковую рандомизацию.

- **«Многорукий бандит»** — динамическая оптимизация в реальной среде: система сама направляет больше трафика на лучший вариант, сокращая число неудач. Полезна для непрерывного улучшения промптов и стратегий поиска.

### Специализированные методы валидации

**Человеческая оценка** — главный стандарт для ответственных применений. Используют структурированные протоколы с чёткими критериями. Методы: согласованность между экспертами, экспертные группы, краудсорсинг.

**Состязательные атаки и безопасность** — проверка устойчивости к:
- быстрым инъекциям,
- отравлению данных,
- извлечению обучающих данных,
- генерации вредного контента.
Применяют автоматические инструменты и ручное тестирование.

**Нагрузочное тестирование** — поведение под высокой нагрузкой:
- максимум одновременных пользователей,
- потребление памяти,
- рост времени ответа,
- стабильность системы.
Инструменты: Apache JMeter, Locust.

**Непрерывный мониторинг в продакшене** — метрики качества, удовлетворённость пользователей, показатели производительности в реальном времени. Для раннего обнаружения деградации — оповещения и автоматический откат.

**Итог:** надёжная RAG-система требует баланса — автоматические тесты для скорости и охвата, человеческая оценка для качества. Валидация охватывает всё: от модульных тестов до A/B-экспериментов.

## 7.3. Инструменты для автоматической оценки

При масштабном внедрении систем на основе LLM необходима автоматическая оценка качества. Современные инструменты предлагают разные подходы — от решений для конкретных задач до платформ, охватывающих весь цикл разработки.

Инструменты оценки быстро развиваются, отражая потребность в объективной валидации ИИ-систем. У каждого свой фокус: от быстрого прототипирования до мониторинга промышленных систем.

### RAGAS — специализированный фреймворк для RAG

RAGAS — самый популярный инструмент для оценки RAG-систем благодаря простоте и пяти ключевым метрикам: точность и полнота контекста, достоверность и релевантность ответа, общая оценка качества.

Настройка занимает 10–15 минут. Фреймворк поддерживает разные языковые модели и формат данных «вопрос–контекст–ответ».

Ограничения: фиксированный набор метрик и отсутствие мониторинга в реальном времени. Подходит для разовой оценки и сравнения конфигураций, но не для непрерывного контроля качества.

### TruLens — универсальная платформа наблюдения

TruLens — комплексная платформа для мониторинга и оценки приложений на базе языковых моделей. Поддерживает более 20 метрик и обеспечивает детальную трассировку запросов через все компоненты.

Ключевое преимущество — продвинутая визуализация: интерактивные дашборды с производительностью в реальном времени, распределением оценок и трендами качества. Интегрируется с популярными фреймворками.

Недостатки: сложная настройка и высокие требования к ресурсам. Требует значительных временных затрат на освоение, поэтому оптимален для крупных корпоративных проектов, а не для малых.

### DeepEval — комплексная система тестирования

DeepEval — лидирующий фреймворк оценки с обширным набором метрик и интеграцией в процессы разработки. Включает более 14 специализированных метрик: обнаружение галлюцинаций, оценку предвзятости, анализ токсичности.

Уникальная особенность — самообъясняющие метрики, которые не только выдают оценку, но и объясняют причины снижения качества. Это упрощает отладку, указывая на конкретные проблемы.

Поддержка модульных тестов и интеграция в процессы непрерывной интеграции делают DeepEval естественным выбором для команд разработки — качество проверяется автоматически при каждом изменении кода.

### Phoenix — мониторинг в реальном времени

Arize Phoenix фокусируется на мониторинге моделей в боевом окружении. Отслеживает деградацию качества, смещения в данных и аномалии в поведении пользователей. Поддерживает анализ трендов и предиктивную аналитику.

Трассировка запросов даёт детальную картину работы сложных систем: каждый запрос отслеживается через все компоненты с замером времени выполнения, потребления ресурсов и промежуточных результатов.

Недостатки: высокие требования к инфраструктуре и сложность развёртывания. Требует специальных знаний для настройки и поддержки, поэтому подходит преимущественно крупным организациям.

### Сравнительная таблица инструментов

| Критерий | RAGAS | TruLens | DeepEval | LlamaIndex | Phoenix | MLFlow | ARES | RAGEval |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Тип инструмента | Специализированный RAG | Универсальная платформа | Комплексная оценка LLM | Встроенная оценка | Мониторинг в реальном времени | ML‑платформа c RAG | Автоматизированная система | Генератор датасетов |
| Основные метрики | 5 ключевых метрик RAG | 20+ универсальных метрик | 15+ метрик БЯМ/RAG | 10+ встроенных метрик | Детекция дрейфа, качество | Стандартные ML + RAG | Context relevance, faithfulness | Сценарно‑специфичные |
| Простота использования | Очень простая | Средняя | Простая | Простая | Сложная | Средняя | Простая | Средняя |
| Время настройки | 10–15 минут | 1–2 часа | 30 минут | 15 минут | 2–4 часа | 1 час | 20 минут | 1 час |
| Документация | Отличная | Хорошая | Отличная | Отличная | Средняя | Хорошая | Хорошая | Средняя |
| Интеграция с фреймворками | Универсальная | LangChain, LlamaIndex | Универсальная | LlamaIndex | Универсальная | MLFlow экосистема | Универсальная | Универсальная |
| Поддержка языков | Многоязычная | Английский преимущественно | Многоязычная | Многоязычная | Многоязычная | Многоязычная | Английский | Многоязычная |
| Визуализация результатов | Базовая | Продвинутая | Хорошая | Встроенная | Продвинутая | Продвинутая | Базовая | Базовая |
| Мониторинг в продакшн | Нет | Да | Ограниченный | Нет | Да | Да | Нет | Нет |
| Кастомизация метрик | Ограниченная | Высокая | Высокая | Средняя | Высокая | Высокая | Ограниченная | Высокая |
| Поддержка A/B‑тестов | Нет | Да | Нет | Нет | Да | Да | Нет | Нет |
| Автоматическое объяснение | Нет | Да | Да | Частично | Да | Нет | Да | Нет |
| Размер сообщества | Большой | Средний | Растущий | Очень большой | Малый | Очень большой | Малый | Малый |
| Лицензия | Apache 2.0 | Apache 2.0 | Apache 2.0 | — | Apache 2.0 | Apache 2.0 | MIT | Apache 2.0 |
| Коммерческая поддержка | Нет | Да | Да | Да | Да | Да | Нет | Нет |
| Требования к ресурсам | — | Средние | Низкие | Низкие | Высокие | Средние | Низкие | Средние |
| Поддержка офлайн‑оценки | Да | Да | Да | Да | Ограниченная | Да | Да | Да |
| Интеграция с CI/CD | Да | Да | Да | Частично | Нет | Да | Нет | Частично |
| Экспорт результатов | JSON, CSV | Множество форматов | JSON, CSV | JSON | Множество форматов | MLFlow формат | JSON | JSON, CSV |
| Идеально для | Быстрая оценка RAG | Полный жизненный цикл | Комплексное тестирование | Разработка с LlamaIndex | Продакшн‑мониторинг | ML‑пайплайны | Исследования | Создание тестовых данных |

### Специализированные решения

ARES — автоматизированная система оценки, минимизирующая ручную разметку. Вместо участия человека использует синтетическую генерацию тестовых сценариев для создания комплексных наборов данных.

RAGEval фокусируется на сценарно-специфичных наборах данных для тестирования. Анализирует предметную область и автоматически генерирует репрезентативные тестовые случаи — критически важно для специализированных применений.

MLFlow добавляет к управлению экспериментами механизм оценки языковых моделей. Интеграция с существующими процессами машинного обучения позволяет командам использовать знакомые инструменты.

**Вывод**

Выбор зависит от потребностей проекта и ресурсов. Для быстрого прототипирования — RAGAS и DeepEval, для крупных продакшн-систем — TruLens и Phoenix, для исследовательских задач — ARES и RAGEval. Понимание сильных и слабых сторон каждого решения позволяет принимать обоснованные архитектурные решения.